# Exploring Tokenizers

My notes on how tokenizers work — the piece that sits between plain text and
the numbers an LLM actually operates on. I use Hugging Face's `AutoTokenizer`
to poke at a few different models (Llama 3.1, Phi-4, DeepSeek, Qwen Coder) and
compare how each one breaks text apart, applies chat templates, and handles code.

**Environment:** this one is light enough to run on a free CPU runtime in
Colab, or locally — no GPU strictly required, unlike the pipelines notebook.


## A couple of things I want to remember about Colab

- Warnings and messages are mostly safe to ignore.
- If I ever see an error like:

  > `Runtime error: CUDA is required but not available for bitsandbytes...`

  this is misleading — it's *not* actually a package version problem. It
  usually means Colab swapped out the runtime underneath me. Fix:
  1. `Runtime` menu -> Disconnect and delete runtime
  2. Reload the notebook fresh, `Edit` menu -> Clear All Outputs
  3. Reconnect to a new T4 (top-right button)
  4. Check "View resources" to confirm the GPU is actually attached
  5. Re-run all cells from the top, starting with the pip installs


## Setup


In [ ]:
!pip install -q --upgrade datasets==3.6.0 transformers==4.57.6


In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer


### Signing in to Hugging Face

1. Free account at https://huggingface.co -> Settings -> create a new API token
   with **write** permissions (important - read-only causes problems later).
2. Add it to Colab's Secrets panel (key icon on the left sidebar) as
   `HF_TOKEN = your_token`, and switch on notebook access.
3. Run the cell below to log in and confirm the GPU (if using one).


In [ ]:
# Log in to Hugging Face

hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

# Check Google Colab GPU

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")


## Getting access to Llama 3.1

Llama 3.1 is gated - Meta requires agreeing to their terms before the model
can be downloaded.

1. Visit https://huggingface.co/meta-llama/Meta-Llama-3.1-8B and follow the
   instructions at the top of the page to request access (using the same
   email as my Hugging Face account).
2. Approval usually comes through within a couple of minutes. Once approved
   for any 3.1 model, it covers the whole 3.1 family.

If the next cell errors out, worth checking:
1. Am I actually logged in to Hugging Face? (`login()` above should confirm the key works)
2. Does my API key have full read/write permissions?
3. Does the model page show I have access, near the top?


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B', trust_remote_code=True)


## Basic tokenization

Encoding a sentence into token IDs.


In [ ]:
text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
tokens


Comparing character count, word count, and token count for the same sentence -
useful for getting a rough intuition of the "tokens per word" ratio for this
tokenizer.


In [ ]:
character_count = len(text)
word_count = len(text.split(' '))
token_count = len(tokens)
print(f"There are {character_count} characters, {word_count} words and {token_count} tokens")


Decoding back from token IDs to text should reconstruct the original string.


In [ ]:
tokenizer.decode(tokens)


`batch_decode` decodes each token ID individually instead of joining them back
into one string - handy for seeing exactly how the text got split up.


In [ ]:
tokenizer.batch_decode(tokens)


Looking at the vocabulary itself: `get_added_vocab()` shows tokens added on
top of the base vocabulary (e.g. special tokens), while `len(tokenizer.vocab)`
gives the total vocabulary size.


In [ ]:
# tokenizer.vocab
tokenizer.get_added_vocab()


In [ ]:
len(tokenizer.vocab)


## Instruct variants of models

Many models have an "Instruct" variant, fine-tuned specifically for chat use.
These expect prompts formatted a particular way, with distinct system/user/
assistant sections.

`apply_chat_template` converts the familiar messages-list format (the same
shape used with the OpenAI API) into the exact prompt string this specific
model expects.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Meta-Llama-3.1-8B-Instruct', trust_remote_code=True)


In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)


## The "aha" moment

Up to now I've been passing a list of Python dictionaries around as if that's
literally what an LLM receives:

```python
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
```

But an LLM is just a statistical model that takes a sequence of numbers and
predicts the probability of the next number - you can't feed Python objects
into that directly.

This is the missing piece: the messages actually get converted through three
stages before reaching the model:

1. Turned into a sequence of words with special tags marking where the
   System / User / Assistant sections begin and end.
2. Those words get broken down into fragments - **tokens**.
3. Each token gets replaced with a **Token ID** - and *that* numeric sequence
   is the real input.

> The input to an LLM is a sequence of Token IDs. The output is a probability
> distribution over what the next Token ID should be.

That's the whole trick - `apply_chat_template` above is just doing step 1 for
me, in whatever exact format this particular model was trained to expect.


## Comparing tokenizers across models

Trying three more models to see how differently each one tokenizes the same
text: Phi-4 (Microsoft), DeepSeek V3.1 (DeepSeek AI), and Qwen2.5-Coder
(Alibaba Cloud) - the last one specifically trained for code.


In [ ]:
PHI4 = "microsoft/Phi-4-mini-instruct"
DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"


In [ ]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print("Llama:")
tokens = tokenizer.encode(text)
print(tokens)
print(tokenizer.batch_decode(tokens))
print("\nPhi 4:")
tokens = phi4_tokenizer.encode(text)
print(tokens)
print(phi4_tokenizer.batch_decode(tokens))


Same sentence, completely different token IDs and splits - each model's
tokenizer was trained on its own vocabulary.

Comparing chat templates too - this is where the differences get even more
visible, since each model wraps the conversation in its own special tokens.


In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi 4:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))


In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"
print(tokenizer.encode(text))
print()
print(phi4_tokenizer.encode(text))
print()
print(deepseek_tokenizer.encode(text))


In [ ]:
print("Llama:")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nPhi:")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
print("\nDeepSeek:")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))


## Tokenizing code

Qwen2.5-Coder is trained specifically on code, so its tokenizer should carve
up source code more sensibly than a general-purpose text tokenizer would -
worth checking token-by-token what it actually does with a small function.


In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)
code_sample = """
def hello_world(person):
  print("Hello", person)
"""
tokens = qwen_tokenizer.encode(code_sample)
for token in tokens:
  print(f"{token}={qwen_tokenizer.decode(token)}")


## Takeaways

- Tokenizers are model-specific - the same sentence produces different token
  IDs on different models, since each was trained with its own vocabulary.
- `apply_chat_template` is what actually bridges the "list of dicts" messages
  format I'm used to and the real string prompt a model consumes - different
  models expect very different formatting here.
- Code-specialized tokenizers (like Qwen2.5-Coder's) handle things like
  indentation and syntax differently than general text tokenizers, since
  they're trained on a corpus dominated by code rather than prose.
